# Transformer

自己动手写一个 Transformer 的 Pytorch 实现，参考了 Annotated Transformer 的结构设计，代码由自己独立实现。

## 1. Model

开始进行模型的实现。

首先看看全局架构。整体核心是一个 `EncoderDecoder` 类，由 `Encoder` 和 `Decoder` 两部分组成，而其自身分别由 `N` 个 `EncoderLayer/DecoderLayer` 堆叠而成。对于 `EncoderLayer`，其包含一个 `MultiHeadAttention` 和 一个 `FeedForward` 层，中间使用了 `AddAndNorm` 层；对于 `DecoderLayer`，其包含两个多头注意力层和一个前馈网络层。我们使用 `SublayerConnection` 封装每个功能层 与 `AddAndNorm` 层的连接。

测试阶段：数据进入 inputs, 得到一个输出，输出加入 outputs，继续下一轮，得到下一个输出... (测试时，Encoder 只需要进行一次计算)

训练阶段：当前内容进入 inputs, 目标内容进入 outputs, GPU 多个线程并行计算...

可以参考下面的图对整体结构与数据流动进行宏观把握：

<div style="display:flex;flex-wrap:wrap;gap:10px;justify-content:center;">
  <img src="assets/transformer_architecture.webp" style="flex:1;min-width:280px;max-width:700px;height:auto;">
  <img src="assets/transformer_data_flow.png" style="flex:1;min-width:280px;max-width:700px;height:auto;">
</div>

### 1.1 整体架构

In [1]:
import torch
import torch.nn as nn
import numpy as np

In [2]:
class HandsonTransformer(nn.Module):
    """ 
    整体结构，包括词嵌入，位置嵌入，EncoderDecoder核心，输出 
    未考虑分词等
    """
    def __init__(self, vocab_size=32000, embed_dim=512, N=6, max_len=5000):
        super().__init__()
        
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.position = Position(embed_dim, max_len)
        self.encoderdecoder = EncoderDecoder(embed_dim, N)
        self.linear = nn.Linear(embed_dim, vocab_size, bias=False)
        self.softmax = nn.Softmax(-1)

        self.linear.weight = self.embedding.weight
        
    def forward(self, inputs, outputs, pad_id=0):

        src_mask = (inputs == pad_id).unsqueeze(1).unsqueeze(2)

        tgt_pad_mask = (outputs == pad_id).unsqueeze(1).unsqueeze(2)
        size = outputs.size(1)
        subsequent_mask = ~torch.tril(torch.ones((1, 1, size, size), device=outputs.device)).bool()
        tgt_mask = tgt_pad_mask | subsequent_mask
        
        inputs_embedded = self.embedding(inputs)
        outputs_embedded = self.embedding(outputs)

        inputs_embedded = self.position(inputs_embedded)
        outputs_embedded = self.position(outputs_embedded)

        output = self.encoderdecoder(inputs_embedded, outputs_embedded, src_mask, tgt_mask)
        output = self.linear(output)
        # output = self.softmax(output)
        
        return output

In [3]:
from math import log

class Position(nn.Module):
    """ Position Embedding """
    def __init__(self, d_model, max_len):
        super().__init__()

        pe = torch.zeros(max_len, d_model)

        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)

        term = torch.exp(torch.arange(0, d_model, 2).float() * (-log(10000.0) / d_model))

        pe[:, 0::2] = torch.sin(position * term)
        pe[:, 1::2] = torch.cos(position * term)

        pe = pe.unsqueeze(0)

        self.register_buffer('pe', pe)
    
    def forward(self, x):
        x = x + self.pe[:, :x.size(1), :]
        return x

### 1.2 核心实现

#### 1.2.1 Encoder and Decoder

In [4]:
class EncoderDecoder(nn.Module):
    """ 核心部件 """
    def __init__(self, embed_dim, N):
        super().__init__()
        
        self.encoder = Encoder(embed_dim, N)
        self.decoder = Decoder(embed_dim, N)

    def forward(self, inputs_embedded, outputs_embedded, src_mask, tgt_mask):
        encoder_out = self.encoder(inputs_embedded, src_mask)
        out = self.decoder(encoder_out, outputs_embedded, src_mask, tgt_mask)
        return out

In [5]:
class Encoder(nn.Module):
    def __init__(self, embed_dim, N):
        super().__init__()
        self.encoders = nn.ModuleList([EncoderLayer(embed_dim) for _ in range(N)])
    
    def forward(self, inputs_embedded, src_mask):
        for encoder in self.encoders:
            inputs_embedded = encoder(inputs_embedded, src_mask)
        return inputs_embedded

In [6]:
class EncoderLayer(nn.Module):
    def __init__(self, embed_dim):
        super().__init__()
        
        self.attention = MultiHeadAttention(8, embed_dim)
        self.ff = FeedForward(embed_dim)

        self.sublayers = nn.ModuleList([SublayerConnection(embed_dim) for _ in range(2)])
        
    def forward(self, inputs_embedded, src_mask):
        x = inputs_embedded
        x = self.sublayers[0](x, lambda _ : self.attention(_, _, _, src_mask))
        x = self.sublayers[1](x, self.ff)
        return x

In [7]:
class SublayerConnection(nn.Module):
    def __init__(self, embed_dim):
        super().__init__()
        
        self.layernorm = nn.LayerNorm(embed_dim)

    def forward(self, x, model):
        x_original = x
        x = model(x)
        x = self.layernorm(x) + x_original
        return x

In [8]:
class Decoder(nn.Module):
    def __init__(self, embed_dim, N):
        super().__init__()
        self.decoders = nn.ModuleList([DecoderLayer(embed_dim) for _ in range(N)])
    
    def forward(self, inputs_embedded, outputs, src_mask, tgt_mask):
        for decoder in self.decoders:
            outputs = decoder(inputs_embedded, outputs, src_mask, tgt_mask)
        return outputs

In [9]:
class DecoderLayer(nn.Module):
    def __init__(self, embed_dim):
        super().__init__()
        
        self.attention1 = MultiHeadAttention(8, embed_dim)
        self.attention2 = MultiHeadAttention(8, embed_dim)
        self.ff = FeedForward(embed_dim)

        self.sublayers = nn.ModuleList([SublayerConnection(embed_dim) for _ in range(3)])
        
    def forward(self, inputs_embedded, current, src_mask, tgt_mask):
        x = current
        x = self.sublayers[0](x, lambda _ : self.attention1(_, _, _, tgt_mask))
        x = self.sublayers[1](x, lambda _ : self.attention2(_, inputs_embedded, inputs_embedded, src_mask))
        x = self.sublayers[2](x, self.ff)
        return x

#### 1.2.2 Attention and FFN

In [10]:
from math import sqrt
from torch.nn.functional import softmax

def attention(Q, K, V, d_k, mask=None):
    x = (Q @ K.transpose(-2, -1)) / sqrt(d_k)

    if mask is not None:
        x += mask * -1e9
    
    x = softmax(x, dim=-1)
    x = x @ V
    return x

In [11]:
class MultiHeadAttention(nn.Module):
    def __init__(self, head, d_model):
        super().__init__()
        assert d_model % head == 0
        d_k = d_model // head
        d_v = d_k

        self.d_k = d_k
        self.d_v = d_v
        self.head = head
        self.d_model = d_model
        
        self.LQ = nn.Linear(d_model, d_model)
        self.LK = nn.Linear(d_model, d_model)
        self.LV = nn.Linear(d_model, d_model)    
        
        self.LO = nn.Linear(d_model, d_model)

    def forward(self, Q, K, V, mask):
        Q = self.LQ(Q)
        K = self.LK(K)
        V = self.LV(V)

        Q = Q.view(Q.size(0), -1, self.head, self.d_k).transpose(1, 2)
        K = K.view(K.size(0), -1, self.head, self.d_k).transpose(1, 2)
        V = V.view(V.size(0), -1, self.head, self.d_v).transpose(1, 2)

        out = attention(Q, K, V, self.d_k, mask)
        out = out.transpose(1, 2).contiguous()
        out = out.view(out.size(0), -1, self.d_model)

        out = self.LO(out)

        return out
        

In [12]:
from torch.nn.functional import relu
class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff=2048):
        super().__init__()

        self.L1 = nn.Linear(d_model, d_ff)
        self.L2 = nn.Linear(d_ff, d_model)

    def forward(self, x):
        x = relu(self.L1(x))
        x = self.L2(x)
        return x

## 2. 简单测试

In [13]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [14]:
is_test = False

In [15]:
if is_test:
    transformer = HandsonTransformer(32000, 512, 6, 5000).to(device)
    
    batch_size = 128
    
    inputs = torch.randint(0, 32000, (batch_size, 100)).to(device)
    outputs = torch.randint(0, 32000, (batch_size, 99)).to(device)
    
    x = transformer(inputs, outputs)
    
    print(x)

## 3. 训练（随机数据）

此部分由 Gemini + 本人共同完成（写完上述架构已经很疲惫了）

In [16]:
import torch.optim as optim

transformer = HandsonTransformer(32000, 512, 6, 5000).to(device)
batch_size = 128

criterion = nn.CrossEntropyLoss(ignore_index=0)

optimizer = optim.Adam(transformer.parameters(), lr=1e-4, betas=(0.9, 0.98), eps=1e-9)

In [17]:
from torch.utils.data import Dataset, DataLoader

class CopyDataset(Dataset):
    def __init__(self, vocab_size, seq_len, num_samples):
        self.vocab_size = vocab_size
        self.seq_len = seq_len
        self.num_samples = num_samples

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        # 随机生成一些 token，避开 0 (pad), 1 (sos), 2 (eos)
        data = torch.randint(3, self.vocab_size, (self.seq_len,))
        
        # Encoder 输入: [data]
        src = data.clone()
        
        # Decoder 输入: [SOS, data] -> 用于 Teacher Forcing
        # 实际标签: [data, EOS] -> 用于计算 Loss
        # 这里为了简化 copy 任务，我们直接让 trg 包含 SOS
        sos = torch.tensor([1])
        eos = torch.tensor([2])
        
        trg = torch.cat([sos, data, eos])
        return src, trg

def collate_fn(batch):
    """
    对 batch 里的数据进行对齐（虽然这里长度固定，但养成好习惯）
    """
    srcs = torch.stack([item[0] for item in batch])
    trgs = torch.stack([item[1] for item in batch])
    return srcs, trgs

In [18]:
# 参数设置
VOCAB_SIZE = 100   # 词汇表大小
SEQ_LEN = 10       # 序列长度
BATCH_SIZE = 32
NUM_SAMPLES = 1000 # 总样本量

dataset = CopyDataset(VOCAB_SIZE, SEQ_LEN, NUM_SAMPLES)
data_loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)

# 测试一下
for src, trg in data_loader:
    print("Source Shape:", src.shape) # [32, 10]
    print("Target Shape:", trg.shape) # [32, 12] (多了 SOS 和 EOS)
    break

Source Shape: torch.Size([32, 10])
Target Shape: torch.Size([32, 12])


In [19]:
if is_test:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = HandsonTransformer(vocab_size=VOCAB_SIZE, embed_dim=128, N=2).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
    criterion = torch.nn.CrossEntropyLoss(ignore_index=0)
    
    for epoch in range(30):
        for src, trg in data_loader:
            src, trg = src.to(device), trg.to(device)
            
            # 核心逻辑：
            # trg_input 是模型看到的: [SOS, data]
            # trg_y 是模型要预测的: [data, EOS]
            trg_input = trg[:, :-1]
            trg_y = trg[:, 1:]
            
            # 前向传播
            output = model(src, trg_input) # 这里会触发你写的 mask 逻辑
            
            # 计算 Loss: (batch * seq_len, vocab_size) vs (batch * seq_len)
            loss = criterion(output.view(-1, VOCAB_SIZE), trg_y.reshape(-1))
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}")

In [20]:
def greedy_decode(model, src, max_len, start_symbol):
    model.eval()
    # src: (1, seq_len)
    device = next(model.parameters()).device
    # 初始化 decoder 输入，起始为 [SOS]
    ys = torch.ones(1, 1).fill_(start_symbol).type_as(src.data).to(device)
    
    for i in range(max_len - 1):
        # 这里的 forward 会调用你写的 mask 逻辑
        with torch.no_grad():
            out = model(src, ys)
        
        # 取最后一个时间步的输出
        prob = out[:, -1]
        _, next_word = torch.max(prob, dim=1)
        next_word = next_word.data[0]
        
        # 拼接新词
        ys = torch.cat([ys, torch.ones(1, 1).type_as(src.data).fill_(next_word)], dim=1)
    return ys

if is_test:
    # 测试一下
    test_src = torch.randint(3, 100, (1, 10)).to(device)
    print("Input Seq:", test_src)
    result = greedy_decode(model, test_src, max_len=11, start_symbol=1)
    print("Output Seq:", result)

## 4. 训练（翻译数据集）

In [21]:
from datasets import load_dataset

dataset = load_dataset("opus100", "en-zh")

In [22]:
print(dataset['train'][0]) 

{'translation': {'en': 'Sixty-first session', 'zh': '第六十一届会议'}}


In [23]:
from transformers import AutoTokenizer
from tqdm import tqdm

# 1. 环境与分词器配置
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_name = "bert-base-multilingual-cased" # 适配中英双语
tokenizer = AutoTokenizer.from_pretrained(model_name)

PAD_ID = tokenizer.pad_token_id
VOCAB_SIZE = tokenizer.vocab_size

# 2. 数据处理函数
def collate_fn(batch):
    # 提取文本
    src_texts = [item['translation']['en'] for item in batch]
    tgt_texts = [item['translation']['zh'] for item in batch]
    
    # 分词并 Padding
    src_encoded = tokenizer(src_texts, padding=True, truncation=True, max_length=128, return_tensors="pt")
    tgt_encoded = tokenizer(tgt_texts, padding=True, truncation=True, max_length=128, return_tensors="pt")
    
    return src_encoded['input_ids'], tgt_encoded['input_ids']

# 3. 训练函数
def train_model(model, train_loader, optimizer, criterion, epochs=1):
    model.train()
    accumulation_steps = 8
    optimizer.zero_grad()
    for epoch in range(epochs):
        loop = tqdm(train_loader, leave=True)
        total_loss = 0
        for i, (src, tgt) in enumerate(loop):
            src, tgt = src.to(device), tgt.to(device)
            
            tgt_input = tgt[:, :-1]
            tgt_y = tgt[:, 1:]
            
            logits = model(src, tgt_input, pad_id=PAD_ID)
            loss = criterion(logits.view(-1, VOCAB_SIZE), tgt_y.reshape(-1))
            display_loss = loss.item()

            loss = loss / accumulation_steps
            
            loss.backward()

            if (i + 1) % accumulation_steps == 0 or (i + 1) == len(train_loader):
                torch.nn.utils.clip_grad_norm_(model.parameters(), 0.5)
                
                optimizer.step()
                optimizer.zero_grad()
            
            total_loss += display_loss
            loop.set_description(f"Epoch {epoch}")
            loop.set_postfix(loss=display_loss)
        
        print(f"Epoch {epoch} finished. Average Loss: {total_loss/len(train_loader):.4f}")

# 4. 推理/测试函数 (Greedy Decode)
def translate(model, sentence, max_len=50):
    model.eval()
    # 编码源句子
    src = tokenizer(sentence, return_tensors="pt")["input_ids"].to(device)
    
    # 初始化解码器输入，以 [CLS] (BERT 的起始符) 开始
    start_symbol = tokenizer.cls_token_id
    ys = torch.ones(1, 1).fill_(start_symbol).type(torch.long).to(device)
    
    for i in range(max_len - 1):
        with torch.no_grad():
            out = model(src, ys, pad_id=PAD_ID)
        
        # 选概率最大的词
        prob = out[:, -1]
        _, next_word = torch.max(prob, dim=1)
        next_word = next_word.item()
        
        # 拼接
        ys = torch.cat([ys, torch.ones(1, 1).type(torch.long).fill_(next_word).to(device)], dim=1)
        
        # 如果生成了 [SEP] (结束符)，停止
        if next_word == tokenizer.sep_token_id:
            break
            
    # 解码为文字
    decoded_text = tokenizer.decode(ys[0], skip_special_tokens=True)
    return decoded_text

# 5. 主程序入口
if __name__ == "__main__":
    # 配置模型参数 (针对 OPUS 任务建议调大 N 和 embed_dim)
    # 使用你之前定义的 HandsonTransformer
    model = HandsonTransformer(
        vocab_size=VOCAB_SIZE, 
        embed_dim=256, 
        N=4, 
        max_len=128 # 确保位置编码够长
    ).to(device)

    # 优化器与损失函数
    optimizer = torch.optim.Adam(model.parameters(), lr=2e-4, betas=(0.9, 0.98), eps=1e-9)
    criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID, label_smoothing=0.1)

    small_train_data = dataset['train'].select(range(50000))
    train_loader = DataLoader(small_train_data, batch_size=8, shuffle=True, collate_fn=collate_fn)

    epochs = 50
    
    for epoch in range(epochs):
        train_model(model, train_loader, optimizer, criterion, epochs=1)
    
        test_sentences = ["The weather is very good today.", 
                          "Hello, world!",
                          "Whereof one cannot speak, thereof one must be silent."]
        
        for test_sentence in test_sentences: 
            print(f"EN: {test_sentence}")
            print(f"ZH: {translate(model, test_sentence)}")
            print("-------------------------------------")

        torch.save(model.state_dict(), f"transformer_epoch_new_{epoch}.pth")

Epoch 0: 100%|██████████████████████████████████████████████████████████████████████████████████████████████| 6250/6250 [09:19<00:00, 11.17it/s, loss=8.76]


Epoch 0 finished. Average Loss: 18.5932
EN: The weather is very good today.
ZH: 我 们 的্রিক্রিক্রিক্রিক্রিক্রিক্রিক্রিক্রিক্রিক্রিক্রিক্রিক্রিক্রিক্রিক্রিক্রিক্রিক্রিক্রিক্রিক্রিক্রিক্রিক্রিক্রিক্রিক্রিক্রিক্রিক্রিক্রিক্রিক্রিক্রিক্রিক্রিক্রিক্রিক্রিক্রিক্রিক্রিক্রিক্রিক
-------------------------------------
EN: Hello, world!
ZH: 好 了
-------------------------------------
EN: Whereof one cannot speak, thereof one must be silent.
ZH: 我 们 的nandonandonandonandonandonandonandonandonandonandonandonandonandonandonandonandonandonandonandonandonandonandonandonandonandonandonandonandonandonandonandonandonandonandonandonandonandonandonandonandonandonandonandonandonandonando
-------------------------------------


Epoch 0: 100%|██████████████████████████████████████████████████████████████████████████████████████████████| 6250/6250 [09:08<00:00, 11.40it/s, loss=6.52]


Epoch 0 finished. Average Loss: 7.2296
EN: The weather is very good today.
ZH: 这 是 很 好 了
-------------------------------------
EN: Hello, world!
ZH: 好 吧 ！
-------------------------------------
EN: Whereof one cannot speak, thereof one must be silent.
ZH: 这 是 ， 有 人 都 是 有 人 都 是 有 人 权 利 于 一 个 人 权 。
-------------------------------------


Epoch 0: 100%|██████████████████████████████████████████████████████████████████████████████████████████████| 6250/6250 [09:05<00:00, 11.46it/s, loss=5.44]


...

Epoch 0 finished. Average Loss: 1.9897
EN: The weather is very good today.
ZH: 马 里 的 结hancehancehance 动
-------------------------------------
EN: Hello, world!
ZH: 你 好 吗 ？
-------------------------------------
EN: Whereof one cannot speak, thereof one must be silent.
ZH: 而 且 在 哪 一 张 不 能 发 生 的 背 而 且 还 是 以 前 使 的 工 作 人 。
-------------------------------------


Epoch 0: 100%|██████████████████████████████████████████████████████████████████████████████████████████████| 6250/6250 [08:41<00:00, 11.99it/s, loss=1.98]


Epoch 0 finished. Average Loss: 1.9767
EN: The weather is very good today.
ZH: 今 天 所 有 乎 很 好 。
-------------------------------------
EN: Hello, world!
ZH: 你 好 ， 世 界 ！
-------------------------------------
EN: Whereof one cannot speak, thereof one must be silent.
ZH: 而 且 在 现 代 表 面 ， 就 不 能 发 生 的 背 横 有 一 种 人
-------------------------------------


Epoch 0: 100%|██████████████████████████████████████████████████████████████████████████████████████████████| 6250/6250 [08:39<00:00, 12.02it/s, loss=2.01]


Epoch 0 finished. Average Loss: 1.9666
EN: The weather is very good today.
ZH: 今 天 要 非 常 好 。
-------------------------------------
EN: Hello, world!
ZH: 你 好
-------------------------------------
EN: Whereof one cannot speak, thereof one must be silent.
ZH: 而 且 在 到 一 件 事 等
-------------------------------------


In [24]:
torch.save(model.state_dict(), "transformer_pf_en_zh.pth")
print("Saved Successfully")

Saved Successfully


## 5. 结语

（写于二六年八月）

几个月前写的 Notebook，当时还处于深度学习的入门阶段。磕磕绊绊按着教程复刻 Transformer，当时虽然最后的实际效果不佳，但还是很开心。几个月来，又学习了好多东西......